## Download Dataset

In [1]:
!gdown "1ENmT6rvuvaDUBeZKLcgY3RQX-1fZVdCr"

Downloading...
From: https://drive.google.com/uc?id=1ENmT6rvuvaDUBeZKLcgY3RQX-1fZVdCr
To: /content/cleaned_vehicles.csv
100% 17.1M/17.1M [00:00<00:00, 53.3MB/s]


## Load Data and Initial Exploration

In [16]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
# Load your file
df = pd.read_csv('cleaned_vehicles.csv')

# Look at your data to identify the target column
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224573 entries, 0 to 224572
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         224573 non-null  int64  
 1   year          224573 non-null  int64  
 2   manufacturer  224573 non-null  object 
 3   model         224573 non-null  object 
 4   cylinders     224573 non-null  object 
 5   fuel          224573 non-null  object 
 6   odometer      224573 non-null  float64
 7   title_status  224573 non-null  object 
 8   transmission  224573 non-null  object 
 9   drive         224573 non-null  object 
 10  type          224573 non-null  object 
 11  paint_color   224573 non-null  object 
 12  state         224573 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 22.3+ MB
None


,price,year,manufacturer,model,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state
0,6995,2000,gmc,new,8,gas,167783.0,clean,automatic,4wd,unknown,red,mn
1,8750,2013,hyundai,sonata,4,gas,90821.0,clean,automatic,fwd,unknown,grey,mn
2,10900,2013,toyota,prius,4,hybrid,92800.0,clean,automatic,fwd,unknown,blue,ct
3,16995,2007,gmc,sierra,8,diesel,254217.0,clean,automatic,4wd,truck,white,mn
4,13995,2012,ford,f-150,6,gas,188406.0,clean,automatic,4wd,truck,grey,mn


## Prepare Original Data for Modeling

In [18]:
# 1. Feature Engineering: Create 'car_age' FIRST
df['car_age'] = 2026 - df['year']

# 2. Define and convert Categorical Columns
cat_cols = ['manufacturer', 'model', 'cylinders', 'fuel', 'title_status',
            'transmission', 'drive', 'type', 'paint_color', 'state']

for col in cat_cols:
    df[col] = df[col].astype('category')

# 3. Log Transform the target
y_log = np.log1p(df['price'])

# 4. Define Features (Drop 'price' and the original 'year')
# We keep 'car_age' and drop 'year' to avoid multi-collinearity
X = df.drop(columns=['price', 'year'])

# 5. Split
X_train, X_test, y_log_train, y_log_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

## Train Model (Original Data)

## General Model (Original Data)

In [19]:
model_log = lgb.LGBMRegressor(
    n_estimators=3000, # Increased because log training is more stable
    learning_rate=0.05,
    num_leaves=63,
    importance_type='gain',
    random_state=42
)

model_log.fit(
    X_train, y_log_train,
    eval_set=[(X_test, y_log_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(200)]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031149 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 673
[LightGBM] [Info] Number of data points in the train set: 179658, number of used features: 12
[LightGBM] [Info] Start training from score 9.403582
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.255764	valid_0's l2: 0.0654151
[400]	valid_0's rmse: 0.249152	valid_0's l2: 0.0620765
[600]	valid_0's rmse: 0.245627	valid_0's l2: 0.0603328
[800]	valid_0's rmse: 0.243042	valid_0's l2: 0.0590693
[1000]	valid_0's rmse: 0.240858	valid_0's l2: 0.0580124
[1200]	valid_0's rmse: 0.239296	valid_0's l2: 0.0572624
[1400]	valid_0's rmse: 0.237914	valid_0's l2: 0.0566032
[1600]	valid_0's rmse: 0.236742	valid_0's l2: 0.0560466
[1800]	valid_0's rmse: 0.235834	valid_0's l2: 0.0556175
[2000]	valid_0's rmse: 0.234965	valid_0's l2: 0.0552084
[2200]	valid_0's rmse: 0.234372	valid_0'

LGBMRegressor(importance_type='gain', learning_rate=0.05, n_estimators=3000,
              num_leaves=63, random_state=42)

## Evaluate Model (Original Data)

In [20]:
# Predict in log scale
log_preds = model_log.predict(X_test)

# Convert back to actual dollars
final_preds = np.expm1(log_preds)
actual_prices = np.expm1(y_log_test)

# New Metrics
from sklearn.metrics import mean_absolute_percentage_error
print(f"New MAE: ${mean_absolute_error(actual_prices, final_preds):.2f}")
print(f"MAPE (Percentage Error): {mean_absolute_percentage_error(actual_prices, final_preds)*100:.2f}%")

New MAE: $1879.84
MAPE (Percentage Error): 15.86%


## Analyze Model Errors (Original Data)

In [21]:
# 1. Create a copy of the test features
results = X_test.copy()

# 2. Convert log values back to actual dollars using np.expm1
results['actual_price'] = np.expm1(y_log_test)
results['predicted_price'] = np.expm1(log_preds) # use your log_preds variable

# 3. Calculate Absolute Error ($) and Percentage Error (%)
results['abs_error_dollars'] = abs(results['actual_price'] - results['predicted_price'])
results['percent_error'] = (results['abs_error_dollars'] / results['actual_price']) * 100

# 4. Sort by the biggest dollar misses
print("--- Top 10 Biggest Hits (Largest Dollar Difference) ---")
display(results.sort_values(by='abs_error_dollars', ascending=False).head(10))

# 5. Sort by the biggest percentage misses (Where the model was 'most confused' relative to price)
print("\n--- Top 10 Biggest Percentage Misses ---")
display(results.sort_values(by='percent_error', ascending=False).head(10))

--- Top 10 Biggest Hits (Largest Dollar Difference) ---


,manufacturer,model,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,car_age,actual_price,predicted_price,abs_error_dollars,percent_error
161520,dodge,charger,8,gas,729.0,clean,automatic,unknown,sedan,unknown,fl,7,99950.0,40543.198243,59406.801757,59.436520
153475,ford,other,8,gas,55055.0,clean,automatic,4wd,pickup,unknown,oh,11,99998.0,43384.738133,56613.261867,56.614394
128932,ford,other,8,gas,55055.0,clean,automatic,4wd,pickup,unknown,oh,11,99998.0,43384.738133,56613.261867,56.614394
92762,mercedes-benz,benz,8,gas,12496.0,clean,automatic,4wd,SUV,yellow,ca,13,87500.0,33192.007637,54307.992363,62.066277
3201,bmw,other,8,gas,9904.0,clean,other,4wd,sedan,grey,dc,7,94500.0,43100.596465,51399.403535,54.390903
112681,chevrolet,other,8,gas,207000.0,clean,automatic,rwd,truck,black,oh,21,55000.0,5592.087862,49407.912138,89.832568
126819,mercedes-benz,other,unknown,gas,7046.0,clean,automatic,unknown,coupe,white,tn,6,97950.0,51325.213116,46624.786884,47.600599
85429,mercedes-benz,other,12,gas,55820.0,clean,automatic,unknown,sedan,white,ny,10,75000.0,29573.218866,45426.781134,60.569042
207173,bmw,other,8,gas,7700.0,clean,automatic,rwd,sedan,grey,ma,11,82995.0,38054.805871,44940.194129,54.148074
157750,bmw,other,unknown,gas,7300.0,clean,automatic,unknown,unknown,grey,ca,8,81999.0,37640.910207,44358.089793,54.095891



--- Top 10 Biggest Percentage Misses ---


,manufacturer,model,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,car_age,actual_price,predicted_price,abs_error_dollars,percent_error
184417,nissan,altima,4,gas,25540.0,clean,automatic,fwd,sedan,grey,ok,9,1000.0,15283.697050,14283.697050,1428.369705
196977,chevrolet,camaro,6,gas,102946.0,clean,automatic,rwd,coupe,black,tn,12,1000.0,15262.337126,14262.337126,1426.233713
82913,chevrolet,camaro,6,gas,102946.0,clean,automatic,rwd,coupe,black,tn,12,1000.0,15262.337126,14262.337126,1426.233713
82916,chevrolet,tahoe,8,gas,126649.0,clean,automatic,rwd,SUV,black,tn,15,1000.0,14799.269757,13799.269757,1379.926976
197074,chevrolet,tahoe,8,gas,126649.0,clean,automatic,rwd,SUV,black,tn,15,1000.0,14799.269757,13799.269757,1379.926976
129654,mercedes-benz,other,6,diesel,48000.0,clean,automatic,unknown,unknown,unknown,tx,11,1950.0,23631.348351,21681.348351,1111.864018
132615,ford,f-150,8,gas,104397.0,clean,automatic,rwd,other,silver,ca,11,1950.0,23466.519705,21516.519705,1103.411267
132616,chevrolet,silverado,8,other,139187.0,clean,automatic,rwd,pickup,black,ca,12,1950.0,21479.549885,19529.549885,1001.515379
128780,kia,forte,4,gas,38843.0,clean,automatic,fwd,sedan,white,tx,7,1000.0,10774.172325,9774.172325,977.417233
135178,toyota,tundra,6,gas,130000.0,clean,automatic,unknown,pickup,silver,fl,16,1499.0,15819.219490,14320.219490,955.318178


## Clean Data (Remove Outliers/Noise)

## Cleaned Data Processing and General Normal Car Predictor

In [24]:
# Create a cleaned version of your dataframe
# 1. Filter out 'placeholder' prices (Down payments)
# 2. Filter out extreme luxury outliers that confuse the general model
df_clean = df[(df['price'] > 2000) & (df['price'] < 85000)].copy()

# 3. Optional: Remove the 'other' model rows if you want higher precision
df_clean = df_clean[df_clean['model'] != 'other']

print(f"Removed {len(df) - len(df_clean)} rows of noise.")

Removed 24895 rows of noise.


## Prepare Cleaned Data for Modeling

In [25]:
# 1. Feature Engineering: Create 'car_age' FIRST
df_clean['car_age'] = 2026 - df_clean['year']

# 2. Define and convert Categorical Columns
cat_cols = ['manufacturer', 'model', 'cylinders', 'fuel', 'title_status',
            'transmission', 'drive', 'type', 'paint_color', 'state']

for col in cat_cols:
    df_clean[col] = df_clean[col].astype('category')

# 3. Log Transform the target
y_log = np.log1p(df_clean['price'])

# 4. Define Features (Drop 'price' and the original 'year')
# We keep 'car_age' and drop 'year' to avoid multi-collinearity
X = df_clean.drop(columns=['price', 'year'])

# 5. Split
X_train, X_test, y_log_train, y_log_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

## Train Model (Cleaned Data)

In [26]:
model_log = lgb.LGBMRegressor(
    n_estimators=10000, # Increased because log training is more stable
    learning_rate=0.01,
    num_leaves=124,
    importance_type='gain',
    random_state=42
)

model_log.fit(
    X_train, y_log_train,
    eval_set=[(X_test, y_log_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(200)]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026211 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 669
[LightGBM] [Info] Number of data points in the train set: 159742, number of used features: 12
[LightGBM] [Info] Start training from score 9.454814
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.248312	valid_0's l2: 0.0616586
[400]	valid_0's rmse: 0.208357	valid_0's l2: 0.0434125
[600]	valid_0's rmse: 0.20083	valid_0's l2: 0.0403326
[800]	valid_0's rmse: 0.197364	valid_0's l2: 0.0389525
[1000]	valid_0's rmse: 0.195491	valid_0's l2: 0.0382166
[1200]	valid_0's rmse: 0.194199	valid_0's l2: 0.0377132
[1400]	valid_0's rmse: 0.19323	valid_0's l2: 0.037338
[1600]	valid_0's rmse: 0.192193	valid_0's l2: 0.0369382
[1800]	valid_0's rmse: 0.191321	valid_0's l2: 0.0366038
[2000]	valid_0's rmse: 0.190507	valid_0's l2: 0.0362928
[2200]	valid_0's rmse: 0.189693	valid_0's l

LGBMRegressor(importance_type='gain', learning_rate=0.01, n_estimators=10000,
              num_leaves=124, random_state=42)

## Evaluate Model (Cleaned Data)

In [27]:
# Predict in log scale
log_preds = model_log.predict(X_test)

# Convert back to actual dollars
final_preds = np.expm1(log_preds)
actual_prices = np.expm1(y_log_test)

# New Metrics
from sklearn.metrics import mean_absolute_percentage_error
print(f"New MAE: ${mean_absolute_error(actual_prices, final_preds):.2f}")
print(f"MAPE (Percentage Error): {mean_absolute_percentage_error(actual_prices, final_preds)*100:.2f}%")

New MAE: $1612.18
MAPE (Percentage Error): 12.10%


## Analyze Model Errors (Cleaned Data)

In [28]:
# 1. Create a copy of the test features
results = X_test.copy()

# 2. Convert log values back to actual dollars using np.expm1
results['actual_price'] = np.expm1(y_log_test)
results['predicted_price'] = np.expm1(log_preds) # use your log_preds variable

# 3. Calculate Absolute Error ($) and Percentage Error (%)
results['abs_error_dollars'] = abs(results['actual_price'] - results['predicted_price'])
results['percent_error'] = (results['abs_error_dollars'] / results['actual_price']) * 100

# 4. Sort by the biggest dollar misses
print("--- Top 10 Biggest Hits (Largest Dollar Difference) ---")
display(results.sort_values(by='abs_error_dollars', ascending=False).head(10))

# 5. Sort by the biggest percentage misses (Where the model was 'most confused' relative to price)
print("\n--- Top 10 Biggest Percentage Misses ---")
display(results.sort_values(by='percent_error', ascending=False).head(10))

--- Top 10 Biggest Hits (Largest Dollar Difference) ---


,manufacturer,model,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,car_age,actual_price,predicted_price,abs_error_dollars,percent_error
205040,ford,econoline,10,gas,16892.0,clean,automatic,rwd,truck,custom,nh,10,74900.0,24590.309178,50309.690822,67.169147
82845,mercedes-benz,sprinter,unknown,diesel,42000.0,clean,automatic,unknown,van,silver,co,11,79900.0,35632.541767,44267.458233,55.403577
74145,gmc,yukon,unknown,gas,238000.0,clean,automatic,4wd,unknown,unknown,in,24,45000.0,4171.888924,40828.111076,90.729136
144923,ram,promaster,unknown,gas,65000.0,clean,automatic,unknown,van,unknown,mi,9,58000.0,17327.719260,40672.280740,70.124622
111261,dodge,durango,6,gas,14500.0,lien,automatic,4wd,SUV,black,co,9,75000.0,34477.787789,40522.212211,54.029616
115773,chevrolet,camaro,8,gas,2500.0,clean,automatic,rwd,coupe,black,mi,8,80500.0,43216.158485,37283.841515,46.315331
137908,jeep,grand,8,gas,30875.0,clean,automatic,4wd,SUV,black,fl,8,76999.0,40938.552678,36060.447322,46.832358
11075,jeep,grand,unknown,gas,45093.0,clean,automatic,unknown,other,unknown,ny,12,54999.0,22076.127315,32922.872685,59.860857
23607,jeep,grand,8,gas,123000.0,clean,automatic,4wd,SUV,black,md,20,38000.0,5890.570251,32109.429749,84.498499
200224,ford,f-150,6,gas,107000.0,clean,automatic,4wd,pickup,red,ne,6,61996.0,30073.436239,31922.563761,51.491328



--- Top 10 Biggest Percentage Misses ---


,manufacturer,model,cylinders,fuel,odometer,title_status,transmission,drive,type,paint_color,state,car_age,actual_price,predicted_price,abs_error_dollars,percent_error
9728,dodge,charger,unknown,gas,30000.0,clean,automatic,unknown,unknown,unknown,fl,8,3000.0,31576.048281,28576.048281,952.534943
8704,mercedes-benz,benz,unknown,gas,65000.0,clean,automatic,4wd,unknown,black,fl,11,2500.0,22912.279148,20412.279148,816.491166
25255,gmc,sierra,8,gas,99995.0,clean,automatic,4wd,truck,white,al,14,3000.0,23121.675591,20121.675591,670.722520
195659,nissan,frontier,6,gas,29475.0,clean,automatic,4wd,pickup,grey,tn,7,4000.0,26853.757129,22853.757129,571.343928
6396,lexus,gs,6,gas,96091.0,clean,automatic,rwd,sedan,silver,fl,13,3000.0,19219.100186,16219.100186,540.636673
198950,ford,transit,6,gas,21800.0,clean,automatic,rwd,van,white,tn,7,4500.0,28738.095659,24238.095659,538.624348
90053,nissan,titan,8,gas,105000.0,clean,automatic,unknown,truck,unknown,tx,13,2500.0,15686.269204,13186.269204,527.450768
6544,nissan,sentra,4,gas,4220.0,clean,automatic,fwd,sedan,grey,fl,9,2500.0,15618.473900,13118.473900,524.738956
198571,ford,transit,6,gas,15435.0,clean,automatic,rwd,van,white,tn,7,4500.0,28041.659690,23541.659690,523.147993
179608,chevrolet,silverado,8,diesel,180928.0,clean,automatic,rwd,truck,white,ca,12,4000.0,24435.089617,20435.089617,510.877240
